# Train Baseline Model

### Imports

In [78]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

from scipy.sparse import hstack

### Load Data from Parquet File

In [70]:
df = pd.read_parquet('data.parquet')
all_games = df['game_id'].unique()
print(f"Unique Games Loaded: {len(all_games)}")
print(f"Total Buckets Loaded: {len(df)}")

print("\nSample Reddit comments:")
comments = df['comments_list'].iloc[1450]
for i, comment in enumerate(comments[:5]):
    print(f"   {comment}")

Unique Games Loaded: 214
Total Buckets Loaded: 51191

Sample Reddit comments:
   I don’t know if I can survive these political commercials. Kill me now.
   We made it fam!
   My girlfriend of five years and I split after five years. Thank god football is back, I need the distraction


### Create Train/Test Split
80% Train, 20% Test

In [71]:
np.random.seed(67)

train_games = np.random.choice(all_games, size=int(0.8 * len(all_games)), replace=False)
test_games = np.array([g for g in all_games if g not in train_games])

train_df = df[df['game_id'].isin(train_games)]
test_df = df[df['game_id'].isin(test_games)]

print(f"Train: {len(train_df)} rows ({len(train_games)} games)")
print(f"Test:  {len(test_df)} rows ({len(test_games)} games)")

Train: 40485 rows (171 games)
Test:  10706 rows (43 games)


### Create Windowed Data
This is so that the model can predict the next price using data from the past $N$ minutes 

In [ ]:
def window_data(df, N=5):
    all_features = []
    
    for _, game_df in df.groupby('game_id'):
        game_df = game_df.sort_values('timestamp')
        if len(game_df) <= N: continue
            
        rows = [
            {
                'text_feature': " </s> ".join(game_df['all_text_clean'].iloc[i-N : i+1]),
                'price_feature': game_df['price_close'].iloc[i-N : i+1].values,
                'count_feature': game_df['comment_count'].iloc[i-N : i+1].values,
                'score_feature': game_df['avg_score'].iloc[i-N : i+1].values,
                'label': game_df['label'].iloc[i]
            }
            for i in range(N, len(game_df))
        ]
        all_features.extend(rows)
        
    return pd.DataFrame(all_features)

### Train Model Using Different $N$ Values
This is so that we can test which value of $N$ results in the best model performance

In [82]:
n_values = [5, 10, 15, 20]
summary_results = []

for n in n_values:
    print(f"[INFO] Training Baseline with N={n}")
    
    train_win = window_data(train_df, N=n)
    test_win = window_data(test_df, N=n)
    
    vec = TfidfVectorizer(max_features=5000)
    X_train_text = vec.fit_transform(train_win['text_feature'])
    X_test_text = vec.transform(test_win['text_feature'])
    
    cols = ['price_feature', 'count_feature', 'score_feature']
    X_train_num = np.hstack([np.vstack(train_win[c]) for c in cols])
    X_test_num = np.hstack([np.vstack(test_win[c]) for c in cols])
    
    scaler = StandardScaler()
    X_train_num_s = scaler.fit_transform(X_train_num)
    X_test_num_s = scaler.transform(X_test_num)
    
    X_train = hstack([X_train_text, X_train_num_s])
    X_test = hstack([X_test_text, X_test_num_s])
    
    model = LogisticRegression(class_weight='balanced', max_iter=1000)
    model.fit(X_train, train_win['label'])
    
    y_pred = model.predict(X_test)
    acc = accuracy_score(test_win['label'], y_pred)
    f1 = f1_score(test_win['label'], y_pred, average='macro')
    
    summary_results.append({'N': n, 'Accuracy': acc, 'F1': f1})

print("\n Training Results")
print(pd.DataFrame(summary_results))


[INFO] Training Baseline with N=5
[INFO] Training Baseline with N=10
[INFO] Training Baseline with N=15
[INFO] Training Baseline with N=20

 Training Results
    N  Accuracy        F1
0   5  0.483271  0.448986
1  10  0.483457  0.453949
2  15  0.470232  0.445671
3  20  0.470039  0.444034
